
# Marine Drifter Prediction with Transformer + Neural ODE

This notebook implements a Physics-Residual Transformer-Neural ODE model for marine drifter trajectory prediction.

Core idea:

\[
\Delta x_{pred}
=
\Delta x_{adv}
+
r_\theta
\]

Where:

- Advection baseline models first-order physical motion
- Transformer extracts temporal dynamics
- Neural ODE models continuous latent evolution
- Residual head learns nonlinear corrections


In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [37]:
!pip install torchdiffeq

import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error

from torchdiffeq import odeint

from tqdm import tqdm


## Load Dataset

In [38]:

def load_year_npz(npz_path):

    data = np.load(npz_path, allow_pickle=True)

    return {
        "X_train": data["X_train"],
        "y_train": data["y_train"],

        "X_val": data["X_val"],
        "y_val": data["y_val"],

        "X_test": data["X_test"],
        "y_test": data["y_test"],

        "feature_cols": data["feature_cols"]
    }

base_dir = "/content/drive/MyDrive/2022-2025_processed_data_extended"

years = [2022, 2023, 2024, 2025]

train_X_list = []
train_y_list = []

val_X_list = []
val_y_list = []

test_X_list = []
test_y_list = []

for year in years:

    npz_path = os.path.join(
        base_dir,
        f"{year}_extended",
        f"drifter_{year}_extended_supervised_windows.npz"
    )

    d = load_year_npz(npz_path)

    train_X_list.append(d["X_train"])
    train_y_list.append(d["y_train"])

    val_X_list.append(d["X_val"])
    val_y_list.append(d["y_val"])

    test_X_list.append(d["X_test"])
    test_y_list.append(d["y_test"])

    feature_cols = d["feature_cols"]

X_train = np.concatenate(train_X_list, axis=0)
y_train = np.concatenate(train_y_list, axis=0)

X_val = np.concatenate(val_X_list, axis=0)
y_val = np.concatenate(val_y_list, axis=0)

X_test = np.concatenate(test_X_list, axis=0)
y_test = np.concatenate(test_y_list, axis=0)

print(X_train.shape)
print(y_train.shape)


(727755, 8, 21)
(727755, 2)


## Feature Selection

In [39]:

selected_features = [
    "latitude",
    "longitude",
    "ve",
    "vn",
    "speed",
    "direction",
    "accel_east",
    "accel_north"
]

feature_cols = list(feature_cols)

feature_idx = [
    feature_cols.index(f)
    for f in selected_features
]

X_train = X_train[:, :, feature_idx]
X_val = X_val[:, :, feature_idx]
X_test = X_test[:, :, feature_idx]

print(X_train.shape)


(727755, 8, 8)


## Advection Baseline

In [40]:

DT_SECONDS = 24 * 3600
METERS_PER_DEG_LAT = 111000

VE_IDX = selected_features.index("ve")
VN_IDX = selected_features.index("vn")

def compute_advection_baseline(X):

    ve = X[:, -1, VE_IDX]
    vn = X[:, -1, VN_IDX]

    delta_lat = (vn * DT_SECONDS) / METERS_PER_DEG_LAT
    delta_lon = (ve * DT_SECONDS) / METERS_PER_DEG_LAT

    baseline = np.stack(
        [delta_lat, delta_lon],
        axis=1
    )

    return baseline

adv_train = compute_advection_baseline(X_train)
adv_val = compute_advection_baseline(X_val)
adv_test = compute_advection_baseline(X_test)

residual_train = y_train - adv_train
residual_val = y_val - adv_val
residual_test = y_test - adv_test


## Dataset

In [41]:

class DrifterDataset(Dataset):

    def __init__(self, X, target_y, adv):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(target_y, dtype=torch.float32)
        self.adv = torch.tensor(adv, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx],
            self.adv[idx]
        )


## Positional Encoding

In [42]:

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=32):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


## Transformer Encoder

In [43]:

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            d_model
        )

        self.pe = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        x = self.input_proj(x)
        x = self.pe(x)

        h = self.encoder(x)

        return h[:, -1]


## Neural ODE

In [44]:

class ODEFunc(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t, z):

        return self.net(z)


class ODEBlock(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.func = ODEFunc(hidden_dim)

    def forward(self, z0):

        t = torch.tensor(
            [0, 1],
            dtype=torch.float32,
            device=z0.device
        )

        z_t = odeint(
            self.func,
            z0,
            t,
            method='rk4'
        )

        return z_t[-1]


## Full Model

In [45]:
class TransformerODEModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.encoder = TransformerEncoder(
            input_dim=input_dim
        )

        self.ode = ODEBlock(128)

        self.head = nn.Sequential(

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        z = self.encoder(x)

        z = self.ode(z)

        residual = self.head(z)

        return residual

## Loss Functions

In [46]:

criterion = nn.MSELoss()

def physics_loss(residual):

    return torch.mean(residual ** 2)

def total_loss(
    pred,
    target,
    residual,
    lambda_phys=0.01
):

    mse = criterion(pred, target)

    phys = physics_loss(residual)

    return mse + lambda_phys * phys


## Training Setup

In [47]:

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = TransformerODEModel(
    input_dim=len(selected_features)
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

print(device)


cuda


## Training Loop

In [48]:
EPOCHS = 100
save_every = 10

## Resume Training from Checkpoint

To allow for continuous training and prevent loss of progress, the training loop will now check for existing checkpoints and resume from the latest one if found. This involves:

1.  **Finding the latest checkpoint**: The code will scan the `checkpoints` directory for files named `model_X.pt` and identify the one with the highest epoch number.
2.  **Loading states**: If a checkpoint is found, the `model`'s state dictionary and the `optimizer`'s state dictionary will be loaded.
3.  **Adjusting starting epoch**: The training loop will then begin from the epoch immediately following the one saved in the checkpoint.

In [49]:
import os

os.makedirs("checkpoints", exist_ok=True)

# Physics Coordinate System (Meters)
DT_SECONDS = 24 * 3600
METERS_PER_DEG = 111000

VE_IDX = selected_features.index("ve")
VN_IDX = selected_features.index("vn")
LAT_IDX = selected_features.index("latitude")


# Convert Targets:
# (delta_north_m, delta_east_m)

def convert_targets_to_meters(X, y):

    lat_now = X[:, -1, LAT_IDX]

    lat_rad = np.deg2rad(lat_now)

    delta_lat = y[:, 0]
    delta_lon = y[:, 1]

    delta_north_m = (
        delta_lat * METERS_PER_DEG
    )

    delta_east_m = (
        delta_lon *
        METERS_PER_DEG *
        np.cos(lat_rad)
    )

    y_meters = np.stack(
        [delta_north_m, delta_east_m],
        axis=1
    )

    return y_meters

y_train_m = convert_targets_to_meters(
    X_train,
    y_train
)

y_val_m = convert_targets_to_meters(
    X_val,
    y_val
)

y_test_m = convert_targets_to_meters(
    X_test,
    y_test
)


##################################################
# Improved Advection Baseline (Meters)
##################################################

def compute_advection_baseline(X):

    ve = X[:, -1, VE_IDX]
    vn = X[:, -1, VN_IDX]

    delta_east_m = ve * DT_SECONDS
    delta_north_m = vn * DT_SECONDS

    baseline = np.stack(
        [delta_north_m, delta_east_m],
        axis=1
    )

    return baseline


adv_train = compute_advection_baseline(X_train)
adv_val = compute_advection_baseline(X_val)
adv_test = compute_advection_baseline(X_test)

residual_train = y_train_m - adv_train
residual_val = y_val_m - adv_val
residual_test = y_test_m - adv_test

residual_mean = residual_train.mean(axis=0)
residual_std = residual_train.std(axis=0)

# avoid divide-by-zero
residual_std = np.where(
    residual_std < 1e-6,
    1.0,
    residual_std
)

##################################################
# Normalize Residuals
##################################################

residual_train_norm = (
    residual_train - residual_mean
) / residual_std

residual_val_norm = (
    residual_val - residual_mean
) / residual_std

residual_test_norm = (
    residual_test - residual_mean
) / residual_std

print("Residual mean:", residual_mean)
print("Residual std:", residual_std)


print("Target unit: meters")
print("Train target shape:", y_train_m.shape)

# Validation Function
def evaluate(model, loader):

    model.eval()

    total_loss_eval = 0

    with torch.no_grad():
        for Xb, yb, advb in loader:

            Xb = Xb.to(device)
            yb = yb.to(device)
            advb = advb.to(device)

            residual_norm = model(Xb)
            residual_physical = (residual_norm * residual_std_tensor + residual_mean_tensor)

            pred = advb + residual_physical

            criterion = nn.SmoothL1Loss()
            loss = criterion(residual_norm,yb)

            total_loss_eval += loss.item()

    return total_loss_eval / len(loader)


optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)


Residual mean: [ -78.44273 1067.8995 ]
Residual std: [73096.06 66969.58]
Target unit: meters
Train target shape: (727755, 2)


In [50]:

train_dataset = DrifterDataset(
    X_train,
    residual_train_norm,
    adv_train
)

val_dataset = DrifterDataset(
    X_val,
    residual_val_norm,
    adv_val
)

test_dataset = DrifterDataset(
    X_test,
    residual_test_norm,
    adv_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128
)


In [51]:
def find_latest_checkpoint(checkpoint_dir="checkpoints"):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("model_") and f.endswith(".pt")]
    if not checkpoints:
        return None

    # Sort by epoch number to get the latest
    checkpoints.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))
    return os.path.join(checkpoint_dir, checkpoints[-1])

latest_checkpoint_path = find_latest_checkpoint()
start_epoch_idx = 0 # 0-indexed epoch to start training from

if latest_checkpoint_path:
    print(f"Resuming training from checkpoint: {latest_checkpoint_path}")
    checkpoint = torch.load(latest_checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    # checkpoint["epoch"] is 1-indexed, so if it's N, it means epoch N was COMPLETED.
    # We should start training from epoch N+1, which is 0-indexed N.
    start_epoch_idx = checkpoint["epoch"]
    print(f"Starting training from epoch {start_epoch_idx + 1}")


# Adjust the training loop to use the `start_epoch_idx`
# The original loop was `for epoch in range(EPOCHS):` where epoch was 0-indexed.
residual_mean_tensor = torch.tensor(
    residual_mean,
    dtype=torch.float32
).to(device)

residual_std_tensor = torch.tensor(
    residual_std,
    dtype=torch.float32
).to(device)

for current_epoch_idx in range(start_epoch_idx, EPOCHS):

    ##############################################
    # Train
    ##############################################

    model.train()
    train_loss = 0
    for Xb, yb, advb in tqdm(train_loader):

      Xb = Xb.to(device)
      yb = yb.to(device)
      advb = advb.to(device)
      optimizer.zero_grad()

      # Predict normalized residual
      residual_norm = model(Xb)
      residual_physical = (residual_norm * residual_std_tensor + residual_mean_tensor)
      criterion = nn.SmoothL1Loss()

      loss =criterion(residual_norm,yb)
      pred = advb + residual_physical
      loss.backward()
      torch.nn.utils.clip_grad_norm_(
          model.parameters(),
          max_norm=1.0
      )
      optimizer.step()

      train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss = evaluate(
        model,
        val_loader
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {current_epoch_idx+1} | " # Print 1-indexed epoch number
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"LR: {current_lr:.2e}"
    )

    if (current_epoch_idx + 1) % save_every == 0:
      torch.save({

          "epoch": current_epoch_idx + 1, # Save as 1-indexed epoch number

          "model_state_dict":
              model.state_dict(),

          "optimizer_state_dict":
              optimizer.state_dict(),

          "train_loss":
              train_loss,

          "val_loss":
              val_loss

      }, f"checkpoints/model_{current_epoch_idx + 1}.pt")


100%|██████████| 5686/5686 [01:13<00:00, 77.31it/s]


Epoch 1 | Train Loss: 0.013960 | Val Loss: 0.008685 | LR: 1.00e-04


100%|██████████| 5686/5686 [01:12<00:00, 78.02it/s]


Epoch 2 | Train Loss: 0.009401 | Val Loss: 0.007934 | LR: 1.00e-04


100%|██████████| 5686/5686 [01:13<00:00, 77.86it/s]


...

Epoch 53 | Train Loss: 0.005427 | Val Loss: 0.005709 | LR: 3.91e-07


100%|██████████| 5686/5686 [01:13<00:00, 77.87it/s]


Epoch 54 | Train Loss: 0.005428 | Val Loss: 0.005706 | LR: 3.91e-07


 45%|████▍     | 2535/5686 [00:32<00:40, 77.94it/s]


KeyboardInterrupt: 

## Evaluation

In [52]:
model.eval()

##################################################
# Load Latest Checkpoint
##################################################

latest_checkpoint_path = find_latest_checkpoint()

if latest_checkpoint_path:

    print(
        f"Loading model from checkpoint "
        f"for evaluation: "
        f"{latest_checkpoint_path}"
    )

    checkpoint = torch.load(
        latest_checkpoint_path
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

else:

    print(
        "No checkpoint found. "
        "Evaluating current model."
    )


##################################################
# Inference
##################################################

preds = []

with torch.no_grad():

    for Xb, _, advb in test_loader:

        Xb = Xb.to(device)

        advb = advb.to(device)

        residual_norm = model(Xb)

        residual_physical = (residual_norm * residual_std_tensor + residual_mean_tensor)

        # Final prediction (meters)
        pred = advb + residual_physical

        preds.append(
            pred.cpu().numpy()
        )

##################################################
# Concatenate Predictions
##################################################

preds = np.concatenate(preds)
targets = y_test_m

##################################################
# Convert Meters -> Lat/Lon Degrees
##################################################

METERS_PER_DEG = 111000

LAT_IDX = selected_features.index(
    "latitude"
)

LON_IDX = selected_features.index(
    "longitude"
)

lat_now = X_test[:, -1, LAT_IDX]

lat_rad = np.deg2rad(lat_now)

##################################################
# Predictions
##################################################

pred_delta_lat = (

    preds[:, 0]

    /

    METERS_PER_DEG
)

pred_delta_lon = (

    preds[:, 1]

    /

    (
        METERS_PER_DEG
        *
        np.cos(lat_rad)
    )
)

##################################################
# Targets
##################################################

target_delta_lat = (

    targets[:, 0]

    /

    METERS_PER_DEG
)

target_delta_lon = (

    targets[:, 1]

    /

    (
        METERS_PER_DEG
        *
        np.cos(lat_rad)
    )
)


##################################################
# RMSE (Degree Space)
##################################################

rmse_lat = np.sqrt(

    mean_squared_error(
        target_delta_lat,
        pred_delta_lat
    )
)

rmse_lon = np.sqrt(

    mean_squared_error(
        target_delta_lon,
        pred_delta_lon
    )
)

print(
    "RMSE Latitude:",
    rmse_lat
)

print(
    "RMSE Longitude:",
    rmse_lon
)


##################################################
# Recover Absolute Future Positions
##################################################

current_lat = X_test[
    :,
    -1,
    LAT_IDX
]

current_lon = X_test[
    :,
    -1,
    LON_IDX
]

pred_future_lat = (
    current_lat
    +
    pred_delta_lat
)

pred_future_lon = (
    current_lon
    +
    pred_delta_lon
)

target_future_lat = (
    current_lat
    +
    target_delta_lat
)

target_future_lon = (
    current_lon
    +
    target_delta_lon
)


##################################################
# Haversine Distance
##################################################

def haversine_km(

    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0

    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)

    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (

        np.sin(dlat / 2) ** 2

        +

        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


##################################################
# Geo Metrics
##################################################

geo_errors = haversine_km(

    target_future_lat,
    target_future_lon,

    pred_future_lat,
    pred_future_lon
)

print(
    "Mean geo (km):",
    np.mean(geo_errors)
)

print(
    "Median geo (km):",
    np.median(geo_errors)
)

print(
    "P90 geo (km):",
    np.percentile(
        geo_errors,
        90
    )
)

Loading model from checkpoint for evaluation: checkpoints/model_50.pt
RMSE Latitude: 0.05880708402395367
RMSE Longitude: 0.7348035819812034
Mean geo (km): 8.60547
Median geo (km): 5.283889
P90 geo (km): 15.024943
